# Data Visualization Gallery — `weather.json`

A hands-on gallery that covers every chart type from the lecture slides:

| Section | Plot types |
|---|---|
| 01 Why visualize? | **Anscombe's Quartet** |
| 02 Distribution | Histogram · KDE overlay · Box plot · Violin |
| 03 Comparison | Bar chart · Grouped bar |
| 04 Relationship | Scatter + regression · Bubble · Correlation heatmap |
| 05 Pair Plot | `seaborn.pairplot` |
| 06 FacetGrid | `seaborn.FacetGrid` |
| 07 Composition & Time | Pie/Donut · Stacked bar · Line · Area |

> Dataset: `weather.json` — 366 daily weather observations (clean version).

## 00 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scipy.stats as sp_stats
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11, "axes.labelsize": 9})

# Consistent color for RainTomorrow
RAIN_PALETTE = {"Yes": "#e41a1c", "No": "#377eb8"}

print("Setup complete.")

## 01 · Load Data

In [ ]:
DATA_PATH = Path("weather.json")
df = pd.read_json(DATA_PATH)

NUM_COLS = df.select_dtypes(include="number").columns.tolist()
CAT_COLS = df.select_dtypes(exclude="number").columns.tolist()

print(f"Shape      : {df.shape}")
print(f"Numeric    : {NUM_COLS}")
print(f"Categorical: {CAT_COLS}")
df.head(3)

---
## Section 01 — Why Visualize? Anscombe's Quartet

> *"The simple graph has brought more information to the data analyst's mind than any other device."* — John Tukey

All four datasets below share **identical** summary statistics (mean, variance, correlation ≈ 0.82) yet look completely different when plotted.  
This is the core motivation for **always visualising before modelling**.

In [ ]:
# --- Anscombe's Quartet data ---
anscombe = {
    "I":   {"x": [10,8,13,9,11,14,6,4,12,7,5],
            "y": [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]},
    "II":  {"x": [10,8,13,9,11,14,6,4,12,7,5],
            "y": [9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]},
    "III": {"x": [10,8,13,9,11,14,6,4,12,7,5],
            "y": [7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]},
    "IV":  {"x": [8,8,8,8,8,8,8,19,8,8,8],
            "y": [6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89]},
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)

for ax, (name, d) in zip(axes, anscombe.items()):
    x, y = np.array(d["x"]), np.array(d["y"])
    r, _ = sp_stats.pearsonr(x, y)
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(min(x) - 1, max(x) + 1, 100)

    ax.scatter(x, y, color="steelblue", s=70, edgecolors="white", lw=0.6, zorder=3)
    ax.plot(xs, m * xs + b, color="tomato", lw=2)
    ax.set_title(
        f"Dataset {name}\n"
        f"$\\bar{{x}}$={np.mean(x):.1f}  "
        f"$\\bar{{y}}$={np.mean(y):.2f}  "
        f"r={r:.2f}",
        fontsize=9,
    )
    ax.set_xlabel("x"); ax.set_ylabel("y")

fig.suptitle(
    "Anscombe's Quartet — Same summary stats, completely different patterns",
    fontsize=12, y=1.03,
)
plt.tight_layout()
plt.show()

---
## Section 02 — Distribution Plots

**When to use:**
- **Histogram** → understand the shape of a single continuous variable; choose bin width carefully.
- **KDE** → smoothed density estimate; useful when the distribution is continuous.
- **Box plot** → compare median, IQR, and outliers across groups.
- **Violin** → combines KDE with box — richer view of distribution shape per group.

In [ ]:
# 2a. Histogram + KDE overlay — effect of bin width
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rain = df["Rainfall"].dropna()
for ax, bins, label in zip(axes, [5, 20, 50], ["bins=5 (too few)", "bins=20 (good)", "bins=50 (too many)"]):
    ax.hist(rain, bins=bins, color="steelblue", alpha=0.55, density=True, edgecolor="white")
    rain.plot.kde(ax=ax, color="tomato", lw=2)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Rainfall (mm)")
    ax.set_ylabel("Density")

fig.suptitle("Histogram + KDE — Effect of Bin Width on Rainfall", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 2b. Multi-variable histogram grid
dist_cols = ["MaxTemp", "MinTemp", "Humidity9am", "Humidity3pm", "Pressure9am", "Temp9am"]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
_legend_handles = [
    mpatches.Patch(color="steelblue", alpha=0.6, label="Hist"),
    plt.Line2D([], [], color="tomato", lw=2, label="KDE"),
    plt.Line2D([], [], color="green", lw=1.5, ls="--", label="Mean"),
    plt.Line2D([], [], color="orange", lw=1.5, ls=":", label="Median"),
]
for ax, col in zip(axes.flat, dist_cols):
    data = pd.to_numeric(df[col], errors="coerce").dropna()
    ax.hist(data, bins=25, color="steelblue", alpha=0.6, density=True, edgecolor="white")
    data.plot.kde(ax=ax, color="tomato", lw=2)
    ax.axvline(data.mean(), color="green", lw=1.5, linestyle="--")
    ax.axvline(data.median(), color="orange", lw=1.5, linestyle=":")
    ax.set_title(f"{col}  (skew={data.skew():.2f})", fontsize=9)
    ax.legend(handles=_legend_handles, fontsize=7, loc="upper right")

fig.suptitle("Histograms + KDE — Continuous Variable Distributions", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 2c. Box plot — compare temperature variables + outlier check
temp_cols = ["MinTemp", "MaxTemp", "Temp9am", "Temp3pm"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: side-by-side box plots
df[temp_cols].dropna().plot.box(ax=axes[0], vert=True, patch_artist=True)
axes[0].set_title("Box Plots — Temperature Variables", fontsize=10)
axes[0].set_ylabel("Temperature (°C)")

# Right: violin + box for Rainfall by RainToday
rain_sub = df[["Rainfall", "RainToday"]].dropna()
sns.violinplot(
    data=rain_sub, x="RainToday", y="Rainfall",
    palette=RAIN_PALETTE, inner="box", ax=axes[1],
)
axes[1].set_title("Violin — Rainfall by RainToday", fontsize=10)
axes[1].set_xlabel("RainToday")
axes[1].set_ylabel("Rainfall (mm)")

plt.tight_layout()
plt.show()

---
## Section 03 — Comparison Plots

**When to use:**
- **Bar chart** → compare a single metric across discrete categories; sort bars for readability.
- **Grouped bar** → compare *multiple* metrics across the same categories.

> ⚠️ Radar/spider charts look impressive but are hard to read accurately — prefer bar charts.

In [ ]:
# 3a. Bar chart — rainy day count per wind direction (sorted)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

rain_by_dir = (
    df[df["RainToday"] == "Yes"]["WindGustDir"]
    .value_counts()
    .sort_values(ascending=False)
)
rain_by_dir.plot.bar(ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Rainy Days by Wind Gust Direction (sorted)", fontsize=10)
axes[0].set_xlabel("Wind Direction")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# 3b. Grouped bar — mean Humidity at 9am vs 3pm by RainTomorrow
grouped = df.groupby("RainTomorrow")[["Humidity9am", "Humidity3pm"]].mean()
grouped.plot.bar(ax=axes[1], edgecolor="white", color=["#4e79a7", "#f28e2b"])
axes[1].set_title("Mean Humidity (9am vs 3pm) by RainTomorrow", fontsize=10)
axes[1].set_xlabel("RainTomorrow")
axes[1].set_ylabel("Humidity (%)")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(fontsize=9)

fig.suptitle("Comparison Plots — Bar & Grouped Bar", fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 04 — Relationship Plots

**When to use:**
- **Scatter + regression** → reveal linear/non-linear associations between two continuous variables.
- **Bubble chart** → encode a third variable via point size.
- **Correlation heatmap** → quick overview of all pairwise linear correlations.

In [ ]:
# 4a. Scatter + regression line for three pairs
pairs = [
    ("Temp9am",    "Temp3pm"),
    ("Pressure9am","Pressure3pm"),
    ("Humidity3pm","Rainfall"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (xc, yc) in zip(axes, pairs):
    sub = df[[xc, yc]].dropna()
    r, _ = sp_stats.pearsonr(sub[xc], sub[yc])
    m, b = np.polyfit(sub[xc], sub[yc], 1)
    xs = np.linspace(sub[xc].min(), sub[xc].max(), 100)

    ax.scatter(sub[xc], sub[yc], alpha=0.35, s=18, color="steelblue", edgecolors="none")
    ax.plot(xs, m * xs + b, color="tomato", lw=2)
    ax.set_xlabel(xc, fontsize=9)
    ax.set_ylabel(yc, fontsize=9)
    ax.set_title(f"r = {r:+.3f}", fontsize=10)

fig.suptitle("Scatter + Regression Lines", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 4b. Bubble chart — Temp3pm vs Humidity3pm, bubble size = WindGustSpeed
sub = df[["Temp3pm", "Humidity3pm", "RainTomorrow"]].copy()
sub["WindGustSpeed"] = pd.to_numeric(df["WindGustSpeed"], errors="coerce")
sub = sub.dropna()
colors = sub["RainTomorrow"].map(RAIN_PALETTE)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    sub["Temp3pm"], sub["Humidity3pm"],
    s=sub["WindGustSpeed"] * 2.5,
    c=colors, alpha=0.55, edgecolors="white", lw=0.4,
)
ax.set_xlabel("Temperature 3pm (°C)")
ax.set_ylabel("Humidity 3pm (%)")
ax.set_title(
    "Bubble Chart — Temp3pm vs Humidity3pm\n"
    "(bubble size ∝ WindGustSpeed, colour = RainTomorrow)"
)
legend_elems = [
    mpatches.Patch(color=RAIN_PALETTE["Yes"], label="RainTomorrow: Yes"),
    mpatches.Patch(color=RAIN_PALETTE["No"],  label="RainTomorrow: No"),
]
ax.legend(handles=legend_elems, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 4c. Correlation heatmap (lower triangle)
corr_cols = [
    "MinTemp", "MaxTemp", "Rainfall",
    "Humidity9am", "Humidity3pm",
    "Pressure9am", "Pressure3pm",
    "Temp9am", "Temp3pm", "WindGustSpeed",
]
# coerce any mixed-type columns (e.g. WindGustSpeed stored as str) to numeric
df_corr = df[corr_cols].apply(pd.to_numeric, errors="coerce")
corr = df_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # hide upper triangle

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, ax=ax,
    annot=True, fmt=".2f",
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    linewidths=0.4,
    annot_kws={"size": 8},
    mask=mask,
)
ax.set_title("Pearson Correlation Heatmap — Weather Variables", fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 05 — Pair Plot (`seaborn.pairplot`)

A **pair plot** is a matrix of scatter plots (off-diagonal) and distributions (diagonal).  
It reveals pairwise relationships and class separation simultaneously — ideal as a first look at a classification dataset.

> **Decision guide**: 4+ continuous variables? → Pair plot.  
> Use `hue` to colour by a categorical label for class overlap insight.

In [ ]:
pp_cols = ["MaxTemp", "Humidity3pm", "WindGustSpeed", "Pressure3pm", "RainTomorrow"]
pp_data = df[pp_cols].copy()
pp_data["WindGustSpeed"] = pd.to_numeric(pp_data["WindGustSpeed"], errors="coerce")
pp_data = pp_data.dropna()

g = sns.pairplot(
    pp_data,
    hue="RainTomorrow",
    diag_kind="kde",
    plot_kws=dict(alpha=0.45, s=20, edgecolor="none"),
    diag_kws=dict(fill=True, alpha=0.5),
    palette=RAIN_PALETTE,
)
g.figure.suptitle(
    "Pair Plot — Weather Variables by RainTomorrow",
    y=1.02, fontsize=12,
)
plt.show()

---
## Section 06 — FacetGrid (`seaborn.FacetGrid`)

A **FacetGrid** repeats the *same* plot for each level of one (or two) categorical variables.  
It is the seaborn way to create **small multiples** — one of the most powerful techniques for comparing subgroups.

> **Rule of thumb**: use a FacetGrid when you want to see whether a pattern holds *across categories*.

In [ ]:
# 6a. FacetGrid — MaxTemp histogram per wind direction (top 8 directions)
top8 = df["WindGustDir"].value_counts().nlargest(8).index
df_top8 = df[df["WindGustDir"].isin(top8)].dropna(subset=["WindGustDir", "MaxTemp"])

g = sns.FacetGrid(
    df_top8, col="WindGustDir", col_wrap=4,
    height=3.2, sharey=False, sharex=True,
)
g.map(sns.histplot, "MaxTemp", bins=15, kde=True, color="steelblue", alpha=0.7)
g.set_titles("{col_name}", size=10)
g.set_axis_labels("MaxTemp (°C)", "Count")
g.figure.suptitle(
    "FacetGrid — MaxTemp Distribution by Wind Gust Direction (Top 8)",
    y=1.03, fontsize=12,
)
plt.show()

In [ ]:
# 6b. FacetGrid — 2D grid: RainToday (rows) × RainTomorrow (cols)
df_rain = df.dropna(subset=["RainToday", "RainTomorrow", "Humidity3pm"])

g2 = sns.FacetGrid(
    df_rain, row="RainToday", col="RainTomorrow",
    height=3, aspect=1.4,
    margin_titles=True,
)
g2.map(sns.kdeplot, "Humidity3pm", fill=True, color="steelblue", alpha=0.55, bw_adjust=1.2)
g2.set_titles(row_template="RainToday={row_name}", col_template="RainTomorrow={col_name}", size=9)
g2.set_axis_labels("Humidity 3pm (%)", "Density")
g2.figure.suptitle(
    "FacetGrid — Humidity 3pm KDE by Rain Status",
    y=1.05, fontsize=12,
)
plt.show()

---
## Section 07 — Composition & Time Charts

**When to use:**
- **Pie / Donut** → part-to-whole for a *small* number of categories (≤ 5); avoid for precise comparisons.
- **Stacked bar** → composition *across multiple groups*; easy to read the bottom category, harder for top ones.
- **Line chart** → trends over time; requires an ordered x-axis.
- **Area chart** → cumulative or stacked magnitudes over time.

In [ ]:
# 7a. Pie and Donut charts
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie — RainTomorrow
rain_counts = df["RainTomorrow"].value_counts()
axes[0].pie(
    rain_counts,
    labels=rain_counts.index,
    autopct="%1.1f%%",
    colors=[RAIN_PALETTE[k] for k in rain_counts.index],
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
)
axes[0].set_title("Pie — RainTomorrow Distribution", fontsize=10)

# Donut — Wind Gust Direction
wd_counts = df["WindGustDir"].value_counts().nlargest(8)
axes[1].pie(
    wd_counts,
    labels=wd_counts.index,
    autopct="%1.0f%%",
    startangle=90,
    wedgeprops=dict(width=0.45, edgecolor="white", linewidth=1),
    textprops={"fontsize": 8},
)
axes[1].set_title("Donut — Top-8 Wind Gust Directions", fontsize=10)

fig.suptitle("Pie / Donut Charts — Part-to-Whole", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 7b. Stacked bar — mean Humidity9am + Humidity3pm by top-8 wind direction
df_top8_humid = (
    df[df["WindGustDir"].isin(top8)]
    .groupby("WindGustDir")[["Humidity9am", "Humidity3pm"]]
    .mean()
    .sort_values("Humidity9am", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
df_top8_humid.plot.bar(stacked=True, ax=ax, edgecolor="white", color=["#4e79a7", "#f28e2b"])
ax.set_title("Stacked Bar — Mean Humidity (9am + 3pm) by Wind Direction", fontsize=10)
ax.set_xlabel("Wind Direction")
ax.set_ylabel("Humidity (%)")
ax.tick_params(axis="x", rotation=0)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 7c. Line chart + Area chart (monthly proxy: divide records into 12 bins)
df_time = df.copy()
df_time["month_bin"] = pd.cut(df_time.index, bins=12, labels=range(1, 13))
monthly = df_time.groupby("month_bin", observed=True)[
    ["MaxTemp", "MinTemp", "Humidity9am", "Humidity3pm", "Rainfall"]
].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Line — temperatures
monthly[["MaxTemp", "MinTemp"]].plot(
    ax=axes[0], marker="o", lw=2, color=["tomato", "steelblue"]
)
axes[0].set_title("Line — Monthly Average Temperature", fontsize=10)
axes[0].set_xlabel("Month bin (proxy)")
axes[0].set_ylabel("°C")
axes[0].legend(["MaxTemp", "MinTemp"], fontsize=9)

# Area — stacked humidity
monthly[["Humidity9am", "Humidity3pm"]].plot.area(
    ax=axes[1], alpha=0.6, color=["#4e79a7", "#f28e2b"]
)
axes[1].set_title("Area — Monthly Avg Humidity (Stacked)", fontsize=10)
axes[1].set_xlabel("Month bin (proxy)")
axes[1].set_ylabel("Humidity (%)")
axes[1].legend(fontsize=9)

fig.suptitle("Line & Area Charts — Temporal Trends", fontsize=12)
plt.tight_layout()
plt.show()

---
## Summary — Choosing the Right Chart

| Goal | Variables | Recommended chart |
|---|---|---|
| Understand one variable | 1 continuous | Histogram + KDE |
| Compare groups | 1 continuous + 1 categorical | Box / Violin |
| Show trend over time | 1 continuous + time axis | Line / Area |
| Reveal association | 2 continuous | Scatter + regression |
| Encode a 3rd dimension | 3 continuous | Bubble |
| Part-to-whole (≤5 cats) | 1 categorical | Pie / Donut |
| Composition across groups | 1 categorical × 1 metric | Stacked bar |
| All pairwise relationships | 4+ continuous | Pair plot |
| Same plot across sub-groups | Any + 1–2 categorical | FacetGrid (small multiples) |

> **Golden rule**: choose the chart that makes the pattern *easiest to read*, not the most impressive-looking one.